In [168]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd().resolve()
PROJECT_ROOT = BASE_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"

print("BASE_DIR:", BASE_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)

years = range(2008, 2023)

price_5min_list = []

for y in years:
    fp = DATA_DIR / f"Y{y}.parquet"
    print("\nLoading:", fp.name)

    df = pd.read_parquet(fp, columns=["price", "YYYYMMDD", "HHMMSS"])

    yyyymmdd = pd.to_numeric(df["YYYYMMDD"], errors="coerce").astype("Int64")
    hhmmss   = pd.to_numeric(df["HHMMSS"], errors="coerce").astype("Int64")
    mask = yyyymmdd.notna() & hhmmss.notna()
    df = df.loc[mask].copy()

    dt_str = yyyymmdd.astype(str) + hhmmss.astype(str).str.zfill(6)
    df["datetime"] = pd.to_datetime(dt_str, format="%Y%m%d%H%M%S", utc=True)

    # tick series within year
    s = df.set_index("datetime").sort_index()["price"]

    # keep last tick if multiple ticks share same timestamp
    n_dups = int(s.index.duplicated().sum())
    if n_dups > 0:
        print(f"  duplicate tick timestamps in {y}: {n_dups} (keeping last)")
    s = s.groupby(level=0).last()

    # resample to 5-min (right-closed / right-labeled)
    p5 = s.resample("5min", label="right", closed="right").last()

    print(f"  year {y}: ticks={len(s):,} -> 5min bins={len(p5):,}, "
          f"missing bins pre-ffill={int(p5.isna().sum()):,}, "
          f"range={p5.index.min()} .. {p5.index.max()}")

    price_5min_list.append(p5)

# concat yearly 5-min series
price_5min = pd.concat(price_5min_list).sort_index()

# handle duplicates at year boundaries (keep last)
n_dups_boundary = int(price_5min.index.duplicated().sum())
print("\nAfter concat: total bins =", len(price_5min))
print("Duplicate timestamps at year boundaries:", n_dups_boundary)
if n_dups_boundary > 0:
    print("Example dup timestamps:", price_5min.index[price_5min.index.duplicated()].unique()[:5])

price_5min = price_5min.groupby(level=0).last()

print("After boundary de-dup: total bins =", len(price_5min))
print("Range:", price_5min.index.min(), "..", price_5min.index.max())

# OPTIONAL: force a complete 5-min grid (creates NaNs on weekends/holidays)
full_idx = pd.date_range(
    start=price_5min.index.min(),
    end=price_5min.index.max(),
    freq="5min",
    tz="UTC",
)
price_5min = price_5min.reindex(full_idx)

print("\nAfter forcing full grid:")
print("  total bins:", len(price_5min))
print("  missing bins created by reindex:", int(price_5min.isna().sum()))

# forward fill to create a continuous 5-min price series
price_5min = price_5min.ffill()

print("\nAfter ffill:")
print("  missing bins:", int(price_5min.isna().sum()))
print("  tz:", price_5min.index.tz)
print("  monotonic:", price_5min.index.is_monotonic_increasing)

# check 5-min spacing (should be strict after reindex)
diffs = price_5min.index.to_series().diff().dropna()
bad = diffs[diffs != pd.Timedelta("5min")]
print("  non-5min gaps:", len(bad))
if len(bad) > 0:
    print("  largest gaps:")
    print(bad.sort_values(ascending=False).head(10))

# sanity preview
print("\nFirst 5 timestamps:", price_5min.index[:5].tolist())
print("Last  5 timestamps:", price_5min.index[-5:].tolist())

price_5min.index.name = "datetime"


BASE_DIR: C:\Users\livsem\master\tio4900_master_thesis\liv
PROJECT_ROOT: C:\Users\livsem\master\tio4900_master_thesis
DATA_DIR: C:\Users\livsem\master\tio4900_master_thesis\data

Loading: Y2008.parquet
  year 2008: ticks=11,058,930 -> 5min bins=105,408, missing bins pre-ffill=31,587, range=2008-01-01 00:05:00+00:00 .. 2009-01-01 00:00:00+00:00

Loading: Y2009.parquet
  year 2009: ticks=8,273,416 -> 5min bins=105,120, missing bins pre-ffill=31,646, range=2009-01-01 00:05:00+00:00 .. 2010-01-01 00:00:00+00:00

Loading: Y2010.parquet
  year 2010: ticks=6,725,347 -> 5min bins=105,096, missing bins pre-ffill=31,601, range=2010-01-01 00:05:00+00:00 .. 2010-12-31 22:00:00+00:00

Loading: Y2011.parquet
  year 2011: ticks=12,564,991 -> 5min bins=104,233, missing bins pre-ffill=30,969, range=2011-01-03 00:00:00+00:00 .. 2011-12-30 22:00:00+00:00

Loading: Y2012.parquet
  year 2012: ticks=11,403,063 -> 5min bins=105,120, missing bins pre-ffill=31,605, range=2012-01-02 00:05:00+00:00 .. 2013-01-01

In [ ]:
# Compute 5-minute log returns
# Using right-labeled 5-min bars, so a return at time t
# represents the interval (t-5min, t]
returns_5min = np.log(price_5min).diff().dropna()
returns_5min.name = "log_return_5min"

# Drop the first 5-min return on Mondays (UTC)
# This removes the "weekend gap" jump from being included in RV

r = returns_5min

# Monday in pandas: 0=Mon, 1=Tue, ..., 6=Sun
is_monday = (r.index.dayofweek == 0)

# position within each calendar day (0=first, 1=second, ...)
pos_in_day = r.groupby(r.index.normalize()).cumcount()

# drop first returns on Mondays
drop_mask = is_monday & (pos_in_day <= 1)

print("Dropping first two Monday 5-min returns:", int(drop_mask.sum()))
print("Example dropped:", r.index[drop_mask][:10].tolist())


 #find first Monday that actually has a dropped return
monday_dropped = r.index[drop_mask]
 
if len(monday_dropped) > 0:
   
    example_ts = monday_dropped[0]
    example_date = example_ts.normalize()
 
    print("Example Monday date:", example_date)
    print("Dropped timestamp:", example_ts)
    print("Dropped return value:", r.loc[example_ts])
   
    # show first 10 returns that day (before dropping)
    print("\nFirst 10 returns that Monday (BEFORE dropping):")
    monday_returns = r.loc[r.index.normalize() == example_date]
    display(monday_returns.head(10))
 
else:
    print("No Monday returns were dropped.")

returns_5min = r.loc[~drop_mask].copy()

if len(monday_dropped) > 0:
    print("\nFirst 10 returns that Monday (AFTER dropping):")
    monday_returns_after = returns_5min.loc[
        returns_5min.index.normalize() == example_date
    ]
    display(monday_returns_after.head(10))


# Construct FX trading day (cutoff at 16:00 UTC) 
# FX day label rule (with right-labeled returns):
# timestamps > 16:00 are assigned to NEXT calendar day;
# timestamps <= 16:00 stay on the same calendar day.
# Returns exactly at 16:00 remain in the current calendar day.

cutoff = pd.to_datetime("16:00:00").time()
idx = returns_5min.index

is_next_day = idx.time > cutoff

fx_day = (
    idx.tz_convert("UTC").normalize() +
    pd.to_timedelta(is_next_day.astype(int), unit="D")
)


# Keep only full FX days (288 returns = 24h × 12 per hour)
counts = returns_5min.groupby(fx_day).size()


# Identify which fx_day labels are Mondays (UTC calendar day)
fx_day_idx = pd.DatetimeIndex(counts.index)
is_fx_monday = (fx_day_idx.dayofweek == 0)

full_days = counts.index[
    ((counts == 288) & (~is_fx_monday)) | ((counts == 286) & is_fx_monday)
]



mask_full = pd.Series(fx_day, index=returns_5min.index).isin(full_days)
returns_full = returns_5min.loc[mask_full].copy()


# --- Align FX-day labels to filtered series ---
fx_day_full = pd.Series(fx_day, index=returns_5min.index).loc[returns_full.index]


# --- Final consistency check ---
counts_full = returns_full.groupby(fx_day_full).size()
print("After filtering: min =", counts_full.min(),
      "max =", counts_full.max(),
      "(#days =", len(counts_full), ")")
print(counts_full.value_counts())


Dropping first two Monday 5-min returns: 1564
Example dropped: [Timestamp('2008-01-07 00:00:00+0000', tz='UTC'), Timestamp('2008-01-07 00:05:00+0000', tz='UTC'), Timestamp('2008-01-14 00:00:00+0000', tz='UTC'), Timestamp('2008-01-14 00:05:00+0000', tz='UTC'), Timestamp('2008-01-21 00:00:00+0000', tz='UTC'), Timestamp('2008-01-21 00:05:00+0000', tz='UTC'), Timestamp('2008-01-28 00:00:00+0000', tz='UTC'), Timestamp('2008-01-28 00:05:00+0000', tz='UTC'), Timestamp('2008-02-04 00:00:00+0000', tz='UTC'), Timestamp('2008-02-04 00:05:00+0000', tz='UTC')]
Example Monday date: 2008-01-07 00:00:00+00:00
Dropped timestamp: 2008-01-07 00:00:00+00:00
Dropped return value: 0.0

First 10 returns that Monday (BEFORE dropping):


datetime
2008-01-07 00:00:00+00:00    0.000000
2008-01-07 00:05:00+00:00    0.000441
2008-01-07 00:10:00+00:00   -0.000129
2008-01-07 00:15:00+00:00    0.000027
2008-01-07 00:20:00+00:00   -0.000090
2008-01-07 00:25:00+00:00    0.000063
2008-01-07 00:30:00+00:00   -0.000088
2008-01-07 00:35:00+00:00   -0.000088
2008-01-07 00:40:00+00:00    0.000058
2008-01-07 00:45:00+00:00    0.000044
Freq: 5T, Name: log_return_5min, dtype: float64


First 10 returns that Monday (AFTER dropping):


datetime
2008-01-07 00:10:00+00:00   -0.000129
2008-01-07 00:15:00+00:00    0.000027
2008-01-07 00:20:00+00:00   -0.000090
2008-01-07 00:25:00+00:00    0.000063
2008-01-07 00:30:00+00:00   -0.000088
2008-01-07 00:35:00+00:00   -0.000088
2008-01-07 00:40:00+00:00    0.000058
2008-01-07 00:45:00+00:00    0.000044
2008-01-07 00:50:00+00:00   -0.000170
2008-01-07 00:55:00+00:00    0.000054
Name: log_return_5min, dtype: float64

After filtering: min = 286 max = 288 (#days = 5477 )
log_return_5min
288    4695
286     782
Name: count, dtype: int64


In [170]:
print("\nFX day label sanity:")
print("returns_5min index tz:", returns_5min.index.tz)
print("fx_day dtype:", pd.Series(fx_day_full).dtype)
print("fx_day sample:", pd.Series(fx_day_full).iloc[:3].tolist())

# check a boundary around 16:00
sample_day = "2016-01-04"
tmp = pd.DataFrame({"ret": returns_5min}).loc[f"{sample_day} 15:55":f"{sample_day} 16:10"].copy()
tmp["fx_day"] = pd.Series(fx_day_full, index=returns_5min.index).loc[tmp.index]
print("\nAround 16:00 UTC:")
print(tmp)



FX day label sanity:
returns_5min index tz: UTC
fx_day dtype: datetime64[ns, UTC]
fx_day sample: [Timestamp('2008-01-02 00:00:00+0000', tz='UTC'), Timestamp('2008-01-02 00:00:00+0000', tz='UTC'), Timestamp('2008-01-02 00:00:00+0000', tz='UTC')]

Around 16:00 UTC:
                                ret                    fx_day
datetime                                                     
2016-01-04 15:55:00+00:00 -0.000203 2016-01-04 00:00:00+00:00
2016-01-04 16:00:00+00:00 -0.001036 2016-01-04 00:00:00+00:00
2016-01-04 16:05:00+00:00  0.000615 2016-01-05 00:00:00+00:00
2016-01-04 16:10:00+00:00 -0.000305 2016-01-05 00:00:00+00:00


In [171]:
d = pd.Timestamp("2008-02-08", tz="UTC")

fx_day_s = pd.Series(fx_day, index=returns_5min.index)          # tz-aware Series
x = returns_full.loc[fx_day_s.loc[returns_full.index] == d]    

print("FX day:", d)
print("first return timestamp:", x.index.min())
print("last  return timestamp:", x.index.max())
print("count:", len(x))

FX day: 2008-02-08 00:00:00+00:00
first return timestamp: 2008-02-07 16:05:00+00:00
last  return timestamp: 2008-02-08 16:00:00+00:00
count: 288


In [172]:
# pick any fx_day that exists
d = pd.Timestamp("2016-01-04", tz="UTC")
fx_day_s = pd.Series(fx_day_full, index=returns_5min.index)

x = returns_5min.loc[fx_day_s == d]
print("FX day:", d)
print("count:", len(x))
print("first:", x.index.min(), "last:", x.index.max())


FX day: 2016-01-04 00:00:00+00:00
count: 287
first: 2016-01-03 16:05:00+00:00 last: 2016-01-04 16:00:00+00:00


In [173]:
# Compute daily realized variance

rv = (
    returns_full
    .groupby(fx_day_full)
    .apply(lambda x: (x**2).sum())
    .to_frame("RV_dec")
)

# Convert to percent-squared units
rv["RV_pct"] = rv["RV_dec"] * 10000

# Compute daily FX close price (last 5-min price in each full FX day)
fx_day_series = pd.Series(fx_day_full, index=returns_full.index)

price_5min_full = price_5min.loc[returns_full.index]  # align prices to the same timestamps as returns_full
daily_close_full = price_5min_full.groupby(fx_day_series).last().rename("close")

# rv index and daily_close_full index should already be tz-aware (UTC)
assert rv.index.tz is not None, "rv.index unexpectedly tz-naive"
assert daily_close_full.index.tz is not None, "daily_close_full.index unexpectedly tz-naive"

rv = rv.merge(daily_close_full.to_frame(), left_index=True, right_index=True, how="left")


print(rv.head())
print(rv.shape)
print("Date range:", rv.index.min(), "to", rv.index.max())

                             RV_dec    RV_pct    close
2008-01-02 00:00:00+00:00  0.000032  0.323124  1.47254
2008-01-03 00:00:00+00:00  0.000034  0.340566  1.47218
2008-01-04 00:00:00+00:00  0.000052  0.517177  1.47736
2008-01-05 00:00:00+00:00  0.000006  0.056157  1.47402
2008-01-06 00:00:00+00:00  0.000000  0.000000  1.47402
(5477, 3)
Date range: 2008-01-02 00:00:00+00:00 to 2022-12-30 00:00:00+00:00


In [174]:
d = pd.Timestamp("2012-01-02", tz="UTC")

# build proper FX-day Series (tz-aware)
fx_day_s = pd.Series(fx_day, index=returns_5min.index)

# extract the 288 returns used in RV
r_day = returns_full.loc[fx_day_s.loc[returns_full.index] == d]

print("FX day:", d)
print("Number of 5-min returns:", len(r_day))
print("First return timestamp:", r_day.index.min())
print("Last return timestamp:", r_day.index.max())


FX day: 2012-01-02 00:00:00+00:00
Number of 5-min returns: 287
First return timestamp: 2012-01-01 16:05:00+00:00
Last return timestamp: 2012-01-02 16:00:00+00:00


In [175]:
print("\nFirst 10 timestamps + returns used:")
print(r_day.iloc[:10])

print("\nLast 10 timestamps + returns used:")
print(r_day.iloc[-10:])




First 10 timestamps + returns used:
datetime
2012-01-01 16:05:00+00:00    0.0
2012-01-01 16:10:00+00:00    0.0
2012-01-01 16:15:00+00:00    0.0
2012-01-01 16:20:00+00:00    0.0
2012-01-01 16:25:00+00:00    0.0
2012-01-01 16:30:00+00:00    0.0
2012-01-01 16:35:00+00:00    0.0
2012-01-01 16:40:00+00:00    0.0
2012-01-01 16:45:00+00:00    0.0
2012-01-01 16:50:00+00:00    0.0
Name: log_return_5min, dtype: float64

Last 10 timestamps + returns used:
datetime
2012-01-02 15:15:00+00:00    0.000120
2012-01-02 15:20:00+00:00    0.000012
2012-01-02 15:25:00+00:00   -0.000085
2012-01-02 15:30:00+00:00   -0.000750
2012-01-02 15:35:00+00:00   -0.000503
2012-01-02 15:40:00+00:00    0.000545
2012-01-02 15:45:00+00:00   -0.000004
2012-01-02 15:50:00+00:00   -0.000217
2012-01-02 15:55:00+00:00   -0.000077
2012-01-02 16:00:00+00:00   -0.000178
Name: log_return_5min, dtype: float64


In [176]:
# make rv index tz-aware (only needs to be done once)
assert rv.index.tz is not None, "rv.index became tz-naive — check fx_day construction"


rv_manual = (r_day ** 2).sum()
d = pd.Timestamp("2012-01-02", tz="UTC")

print("Manual RV_dec:", rv_manual)
print("Stored RV_dec:", rv.loc[d, "RV_dec"])
print("Difference:", rv_manual - rv.loc[d, "RV_dec"])



Manual RV_dec: 1.1075577815333471e-05
Stored RV_dec: 1.1075577815333471e-05
Difference: 0.0


In [177]:
# Annualized volatility in percent
rv["RVOL_pct_ann"] = np.sqrt(rv["RV_pct"] * 252)

print(rv.head())


                             RV_dec    RV_pct    close  RVOL_pct_ann
2008-01-02 00:00:00+00:00  0.000032  0.323124  1.47254      9.023702
2008-01-03 00:00:00+00:00  0.000034  0.340566  1.47218      9.264052
2008-01-04 00:00:00+00:00  0.000052  0.517177  1.47736     11.416154
2008-01-05 00:00:00+00:00  0.000006  0.056157  1.47402      3.761855
2008-01-06 00:00:00+00:00  0.000000  0.000000  1.47402      0.000000


In [178]:
print("Mean RV_dec:", rv["RV_dec"].mean())
print("Mean RV_pct:", rv["RV_pct"].mean())

Mean RV_dec: 2.5951760829719765e-05
Mean RV_pct: 0.25951760829719767


In [ ]:

MEDRV_SCALE = np.pi / (6 - 4*np.sqrt(3) + np.pi)

def medrv_day(r: pd.Series) -> float:
    a = np.abs(r.to_numpy())
    N = len(a)
    if N < 3:
        return np.nan
    if N not in (288, 287):
        print("Warning: Expected 288 (normal) or 287 (FX-Monday) returns, got", N)
    med3 = np.median(np.vstack([a[:-2], a[1:-1], a[2:]]), axis=0)
    return MEDRV_SCALE * (N / (N - 2)) * np.sum(med3**2)

cc = returns_full.groupby(fx_day_full).apply(medrv_day).to_frame("CC_dec")
cc["CC_pct"] = cc["CC_dec"] * 10000

print("CC head:")
display(cc.head())


CC head:


,CC_dec,CC_pct
2008-01-02 00:00:00+00:00,0.000024,0.244325
2008-01-03 00:00:00+00:00,0.000030,0.304615
2008-01-04 00:00:00+00:00,0.000021,0.206842
2008-01-05 00:00:00+00:00,0.000006,0.064205
2008-01-06 00:00:00+00:00,0.000000,0.000000


In [ ]:


# Median realized quarticity MedRQ_t

MEDRQ_CONST = (3 * np.pi) / (9*np.pi + 72 - 52*np.sqrt(3))

def medrq_day(r: pd.Series) -> float:
    a = np.abs(r.to_numpy())
    N = len(a)
    if N < 3:
        return np.nan

    # rolling median of 3 adjacent abs returns: length N-2
    med3 = np.median(np.vstack([a[:-2], a[1:-1], a[2:]]), axis=0)

    # formula: const * N * (N/(N-2)) * sum(med3^4)
    return MEDRQ_CONST * N * (N / (N - 2)) * np.sum(med3**4)

medrq = returns_full.groupby(fx_day_full).apply(medrq_day).to_frame("MedRQ")

print("MedRQ head:")
display(medrq.head())

MedRQ head:


,MedRQ
2008-01-02 00:00:00+00:00,1.194755e-09
2008-01-03 00:00:00+00:00,2.332182e-09
2008-01-04 00:00:00+00:00,1.568284e-09
2008-01-05 00:00:00+00:00,1.763387e-10
2008-01-06 00:00:00+00:00,0.000000e+00


In [ ]:

#  Positive/Negative semivariance + Signed measure SJ
r = returns_full
g = fx_day_full

PV = r.where(r >= 0).pow(2).groupby(g).sum().to_frame("PV_dec")
NV = r.where(r < 0).pow(2).groupby(g).sum().to_frame("NV_dec")

sv = PV.join(NV, how="inner")
sv["PV_pct"] = sv["PV_dec"] * 10000
sv["NV_pct"] = sv["NV_dec"] * 10000

sv["SJ_dec"] = sv["PV_dec"] - sv["NV_dec"]
sv["SJ_pct"] = sv["SJ_dec"] * 10000

print("PV/NV/SJ head:")
display(sv.head())

# sanity check: RV ≈ PV + NV
tmp = rv.join(sv, how="inner")
print("max |RV - (PV+NV)| =", (tmp["RV_dec"] - (tmp["PV_dec"] + tmp["NV_dec"])).abs().max())

PV/NV/SJ head:


,PV_dec,NV_dec,PV_pct,NV_pct,SJ_dec,SJ_pct
2008-01-02 00:00:00+00:00,0.000021,0.000012,0.206141,0.116983,8.915862e-06,0.089159
2008-01-03 00:00:00+00:00,0.000015,0.000019,0.148640,0.191926,-4.328548e-06,-0.043285
2008-01-04 00:00:00+00:00,0.000042,0.000010,0.420346,0.096831,3.235155e-05,0.323516
2008-01-05 00:00:00+00:00,0.000002,0.000003,0.023657,0.032500,-8.842918e-07,-0.008843
2008-01-06 00:00:00+00:00,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000


max |RV - (PV+NV)| = 1.0842021724855044e-19


In [182]:
# Build one unified daily table (no duplicate columns)

daily = rv.copy()  # contains RV_dec, RV_pct, RVOL_pct_ann

# join CC
daily = daily.join(cc[["CC_dec", "CC_pct"]], how="inner")

# JC
daily["JC_dec"] = np.maximum(daily["RV_dec"] - daily["CC_dec"], 0.0)
daily["JC_pct"] = daily["JC_dec"] * 10000

# join MedRQ
daily = daily.join(medrq[["MedRQ"]], how="inner")

# CQ = CC * sqrt(MedRQ)
daily["CQ"] = daily["CC_dec"] * np.sqrt(daily["MedRQ"])

# join PV/NV/SJ
daily = daily.join(sv[[
    "PV_dec", "NV_dec", "SJ_dec",
    "PV_pct", "NV_pct", "SJ_pct"
]], how="inner")

In [183]:
# Final sanity checks

print("Final daily shape:", daily.shape)
print("Date range:", daily.index.min(), "to", daily.index.max())
print("Any duplicate columns?", daily.columns.duplicated().any())

max_diff = (daily["RV_dec"] - (daily["PV_dec"] + daily["NV_dec"])).abs().max()
print("max |RV - (PV+NV)| =", max_diff)

display(daily.head())

Final daily shape: (5477, 16)
Date range: 2008-01-02 00:00:00+00:00 to 2022-12-30 00:00:00+00:00
Any duplicate columns? False
max |RV - (PV+NV)| = 1.0842021724855044e-19


,RV_dec,RV_pct,close,RVOL_pct_ann,CC_dec,CC_pct,JC_dec,JC_pct,MedRQ,CQ,PV_dec,NV_dec,SJ_dec,PV_pct,NV_pct,SJ_pct
2008-01-02 00:00:00+00:00,0.000032,0.323124,1.47254,9.023702,0.000024,0.244325,0.000008,0.078799,1.194755e-09,8.445137e-10,0.000021,0.000012,8.915862e-06,0.206141,0.116983,0.089159
2008-01-03 00:00:00+00:00,0.000034,0.340566,1.47218,9.264052,0.000030,0.304615,0.000004,0.035951,2.332182e-09,1.471069e-09,0.000015,0.000019,-4.328548e-06,0.148640,0.191926,-0.043285
2008-01-04 00:00:00+00:00,0.000052,0.517177,1.47736,11.416154,0.000021,0.206842,0.000031,0.310334,1.568284e-09,8.191281e-10,0.000042,0.000010,3.235155e-05,0.420346,0.096831,0.323516
2008-01-05 00:00:00+00:00,0.000006,0.056157,1.47402,3.761855,0.000006,0.064205,0.000000,0.000000,1.763387e-10,8.525976e-11,0.000002,0.000003,-8.842918e-07,0.023657,0.032500,-0.008843
2008-01-06 00:00:00+00:00,0.000000,0.000000,1.47402,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000


In [199]:
# --- Load allowed dates (weekends/holidays already removed) ---
dates_only = pd.read_csv(DATA_DIR / "dates_only.csv")

# ensure these are plain python dates
dates_only["date"] = pd.to_datetime(dates_only["datetime"]).dt.date
cal = dates_only[["date"]].drop_duplicates().set_index("date").sort_index()

# ensure daily index is ALSO plain python dates (fixes tz mismatch)
daily_dates = daily.copy()
daily_dates.index = pd.to_datetime(daily_dates.index).date

daily_cal = cal.join(daily_dates, how="left")

print("After calendar join:", daily_cal.shape)
print("Missing daily rows (no data for a calendar date):", int(daily_cal["RV_dec"].isna().sum()))

# Drop days that are missing (no daily data)
daily_cal = daily_cal.dropna(subset=["RV_dec"])

# Drop days with ~zero RV (optional)
tol = 1e-20
daily_cal = daily_cal.loc[daily_cal["RV_dec"].abs() > tol].copy()

print("After dropping missing + zero-RV days:", daily_cal.shape)
print("New date range:", daily_cal.index.min(), "to", daily_cal.index.max())


After calendar join: (3741, 16)
Missing daily rows (no data for a calendar date): 0
After dropping missing + zero-RV days: (3741, 16)
New date range: 2008-01-02 to 2022-12-30


In [200]:
print(daily_cal.head(5))

# Sort by the filtered date sequence
main_frame = daily_cal.copy()
main_frame = main_frame.sort_index()

# --- price based daily log return computed after filtering ---
main_frame["return_unshifted"] = main_frame["close"].pct_change()      # simple return
main_frame["log_return_unshifted"] = np.log(main_frame["close"]).diff()

# drop first row (since diff is NaN)
main_frame = main_frame.dropna(subset=["return_unshifted", "log_return_unshifted"]).copy()


print(main_frame.shape)

              RV_dec    RV_pct     close  RVOL_pct_ann    CC_dec    CC_pct  \
date                                                                         
2008-01-02  0.000032  0.323124  1.472540      9.023702  0.000024  0.244325   
2008-01-03  0.000034  0.340566  1.472180      9.264052  0.000030  0.304615   
2008-01-04  0.000052  0.517177  1.477360     11.416154  0.000021  0.206842   
2008-01-07  0.000017  0.172626  1.471500      6.595583  0.000014  0.140322   
2008-01-08  0.000023  0.232091  1.471375      7.647669  0.000020  0.200050   

              JC_dec    JC_pct         MedRQ            CQ    PV_dec  \
date                                                                   
2008-01-02  0.000008  0.078799  1.194755e-09  8.445137e-10  0.000021   
2008-01-03  0.000004  0.035951  2.332182e-09  1.471069e-09  0.000015   
2008-01-04  0.000031  0.310334  1.568284e-09  8.191281e-10  0.000042   
2008-01-07  0.000003  0.032304  6.734169e-10  3.641385e-10  0.000007   
2008-01-08  0.000003 

In [ ]:
# Make sure index is sorted
main_frame = main_frame.sort_index()

# Weekly (5 trading days) and Monthly (22 trading days) realized variance
main_frame["RV_W_dec"] = main_frame["RV_dec"].rolling(5).mean()
main_frame["RV_M_dec"] = main_frame["RV_dec"].rolling(22).mean()

# volatility-scale versions (daily decimal vol):
main_frame["RV_W_vol"] = np.sqrt(main_frame["RV_W_dec"])
main_frame["RV_M_vol"] = np.sqrt(main_frame["RV_M_dec"])

# 1-step-ahead targets (tomorrow's return on today's row)
main_frame["log_return"] = main_frame["log_return_unshifted"].shift(-1)
main_frame["return"]     = main_frame["return_unshifted"].shift(-1)

# last row has no t+1 target -> drop it
main_frame = main_frame.dropna(subset=["log_return", "return"]).copy()

# Drop rows where weekly/monthly RV are not available
main_frame = main_frame.dropna(subset=["RV_W_dec", "RV_M_dec"]).copy()

In [213]:
eps = 1e-30  # tiny constant for numerical safety

# --- Variance-type measures (units r^2) -> volatility-scale (units r) ---
main_frame["RV_vol"] = np.sqrt(np.maximum(main_frame["RV_dec"], 0.0))
main_frame["CC_vol"] = np.sqrt(np.maximum(main_frame["CC_dec"], 0.0))
main_frame["JC_vol"] = np.sqrt(np.maximum(main_frame["JC_dec"], 0.0))
main_frame["PV_vol"] = np.sqrt(np.maximum(main_frame["PV_dec"], 0.0))
main_frame["NV_vol"] = np.sqrt(np.maximum(main_frame["NV_dec"], 0.0))

# Vol-scale asymmetry
main_frame["SJ_vol"] = main_frame["PV_vol"] - main_frame["NV_vol"]

# --- Quarticity-type measures (units r^4) -> return-scale (units r) via 4th root ---
main_frame["MedRQ_4root"] = np.power(np.maximum(main_frame["MedRQ"], 0.0), 0.25)
main_frame["CQ_4root"]    = np.power(np.maximum(main_frame["CQ"], 0.0), 0.25)


In [214]:
from scipy.stats import skew, kurtosis

# --- Summary table ---
vars_to_summarize = [
    "log_return", "return",
    # original (variance scale)
    "RV_dec", "RV_W_dec", "RV_M_dec", "PV_dec", "NV_dec", "CC_dec", "JC_dec", "SJ_dec", "CQ", 

    # new (return/vol scale)
    "RV_vol", "RV_W_vol", "RV_M_vol", "PV_vol", "NV_vol", "CC_vol", "JC_vol", "SJ_vol",
    "CQ_4root",
]

summary_rows = []

for col in vars_to_summarize:
    x = main_frame[col].dropna()

    mean = x.mean()
    std = x.std()
    sk = skew(x, bias=False)
    kurt = kurtosis(x, fisher=False, bias=False)  
    max_ = x.max()
    min_ = x.min()
    rho1 = x.autocorr(lag=1)
    rho5 = x.autocorr(lag=5)
    rho22 = x.autocorr(lag=22)

    summary_rows.append([mean, std, sk, kurt, max_, min_, rho1, rho5, rho22])

summary_table = pd.DataFrame(
    summary_rows,
    index=vars_to_summarize,
    columns=["Mean", "Std", "Skewness", "Kurtosis", "Max", "Min", "rho_1", "rho_5", "rho_22"]
)

summary_table

,Mean,Std,Skewness,Kurtosis,Max,Min,rho_1,rho_5,rho_22
log_return,-8.849531e-05,5.984166e-03,0.131611,6.710560,4.617150e-02,-3.844817e-02,0.016877,-0.008623,0.005511
return,-7.058762e-05,5.986384e-03,0.182988,6.815132,4.725400e-02,-3.771842e-02,0.016678,-0.008632,0.005330
RV_dec,3.487871e-05,3.808089e-05,4.515784,38.967663,6.138734e-04,1.041809e-08,0.705740,0.635983,0.460769
RV_W_dec,3.488579e-05,3.234484e-05,3.253399,18.489603,3.229686e-04,2.929742e-06,0.979818,0.834543,0.662203
RV_M_dec,3.488155e-05,2.957716e-05,2.826498,13.788560,2.227655e-04,4.156133e-06,0.998151,0.979103,0.816509
PV_dec,1.751553e-05,2.065742e-05,5.734318,65.294059,4.016736e-04,4.801792e-09,0.603246,0.526926,0.379585
NV_dec,1.736318e-05,1.944931e-05,4.289946,35.264812,3.099075e-04,2.255215e-09,0.682632,0.621126,0.457684
CC_dec,3.061527e-05,3.375373e-05,4.448970,36.481956,4.995244e-04,5.746985e-09,0.727089,0.663366,0.477030
JC_dec,4.518233e-06,8.562048e-06,7.847322,116.749380,2.019480e-04,0.000000e+00,0.115595,0.165348,0.107646
SJ_dec,1.523465e-07,1.264338e-05,2.556460,72.755183,1.963721e-04,-1.770461e-04,0.049173,-0.016934,0.010626


In [215]:
# --- Save ---
out_csv = DATA_DIR / "daily_variance_measures_filtered.csv"
main_frame.reset_index(names="date").to_csv(out_csv, index=False)
print("Saved:", out_csv)

Saved: C:\Users\livsem\master\tio4900_master_thesis\data\daily_variance_measures_filtered.csv


In [217]:
EPS = 1e-12

# --- Load raw IV CSV ---
iv_raw = pd.read_csv(DATA_DIR / "IV_surface_raw.csv")

# Parse IV date (it is like 20070102)
iv_raw["date"] = pd.to_datetime(iv_raw["date"].astype(str), format="%Y%m%d", errors="coerce")

# select the columns  
iv_cols = [
    "D_1_Put_50", "D_1_Call_50",
    "W_1_Put_50", "W_1_Call_50",
    "M_1_Put_50", "M_1_Call_50",
]

missing = [c for c in iv_cols if c not in iv_raw.columns]
if missing:
    raise ValueError(f"Missing expected IV columns: {missing}")

iv = iv_raw[["date"] + iv_cols].copy()

# Ensure numeric
for c in iv_cols:
    iv[c] = pd.to_numeric(iv[c], errors="coerce")

# --- Align to trading calendar (same dates as RV) ---
dates = pd.read_csv(DATA_DIR / "dates_only.csv")
dates["date"] = pd.to_datetime(dates["datetime"])
dates = dates[["date"]].drop_duplicates().sort_values("date")

iv = dates.merge(iv, on="date", how="left")

#  drop days where any of the required IV columns are missing
iv = iv.dropna(subset=iv_cols, how="any").copy()

for c in iv_cols:
    iv[f"IVvol_daily_{c}"] = (iv[c] / 100.0) / np.sqrt(252.0)

# Create ATM average volatility per maturity
iv["IVvol_daily_D_1_ATM"] = (
    iv["IVvol_daily_D_1_Put_50"] +
    iv["IVvol_daily_D_1_Call_50"]
) / 2

iv["IVvol_daily_W_1_ATM"] = (
    iv["IVvol_daily_W_1_Put_50"] +
    iv["IVvol_daily_W_1_Call_50"]
) / 2

iv["IVvol_daily_M_1_ATM"] = (
    iv["IVvol_daily_M_1_Put_50"] +
    iv["IVvol_daily_M_1_Call_50"]
) / 2


iv_clean = iv[
    ["date",
     "IVvol_daily_D_1_ATM",
     "IVvol_daily_W_1_ATM",
     "IVvol_daily_M_1_ATM"]
].copy()


iv_clean.to_csv(DATA_DIR / "IV_ATM_daily_vol_dec.csv", index=False)

print("Saved: IV_ATM_daily_vol_dec.csv")
print(iv_clean.head())
print(iv_clean.isna().mean())

Saved: IV_ATM_daily_vol_dec.csv
        date  IVvol_daily_D_1_ATM  IVvol_daily_W_1_ATM  IVvol_daily_M_1_ATM
0 2008-01-02             0.006614             0.006221             0.006157
1 2008-01-03             0.009453             0.006252             0.006078
2 2008-01-04             0.003467             0.006051             0.005938
3 2008-01-07             0.007874             0.005827             0.006126
4 2008-01-08             0.007874             0.005512             0.005693
date                   0.0
IVvol_daily_D_1_ATM    0.0
IVvol_daily_W_1_ATM    0.0
IVvol_daily_M_1_ATM    0.0
dtype: float64


In [ ]:
# main_frame index -> column
rv_measures = main_frame.reset_index().rename(columns={"index": "date"}).copy()

# rv_measures = main_frame.reset_index().copy()

# Ensure datetime type
rv_measures["date"] = pd.to_datetime(rv_measures["date"])

iv_clean["date"] = pd.to_datetime(iv_clean["date"])

joint = rv_measures.merge(iv_clean, on="date", how="inner")

print("joint shape:", joint.shape)
print("date range:", joint["date"].min(), "to", joint["date"].max())
print("Missing IV columns share:\n", joint[[
    "IVvol_daily_D_1_ATM","IVvol_daily_W_1_ATM","IVvol_daily_M_1_ATM"
]].isna().mean())


joint shape: (3713, 36)
date range: 2008-02-01 00:00:00 to 2022-12-28 00:00:00
Missing IV columns share:
 IVvol_daily_D_1_ATM    0.0
IVvol_daily_W_1_ATM    0.0
IVvol_daily_M_1_ATM    0.0
dtype: float64


In [219]:
# Make sure log_return exists
if "log_return" not in joint.columns:
    raise ValueError("log_return not found in joint. Check that it was created in main_frame before merging.")

for c in ["log_return_unshifted", "return_unshifted"]:
    if c not in joint.columns:
        raise ValueError(f"{c} not found in joint. Check shifting step.")

# Reorder: date, log_return, then everything else
first_cols = ["date", "log_return", "log_return_unshifted", "return", "return_unshifted", "close", "RV_dec", "RV_vol", "RV_W_dec", "RV_W_vol", "RV_M_dec", "RV_M_vol", "CC_dec", "CC_vol", "JC_dec", "JC_vol", "PV_dec", "PV_vol", "NV_dec", "NV_vol", "SJ_dec", "SJ_vol", "CQ", "CQ_4root", "IVvol_daily_D_1_ATM", "IVvol_daily_W_1_ATM", "IVvol_daily_M_1_ATM"]
joint = joint[first_cols]


In [220]:
joint.to_csv(DATA_DIR / "RV_IV_joint.csv", index=False)

print("Saved: RV_IV_joint.csv")


Saved: RV_IV_joint.csv


## Combination 1
 
<small>
IV and RV daily, decimal<br>
RV and others are variance, not volatility
</small>

In [ ]:
# =========================
# COMBO 1
# Base: joint
# Includes: date, close, log_return,
#           RV (daily/weekly/monthly variance, decimal),
#           other realized variance measures (decimal),
#           IV daily vols (decimal)
# =========================
 
combo1 = joint.copy()
 
# Rename realized-variance columns to standardized names
rename_map = {
    "RV_dec": "RV",
    "RV_W_dec": "RV_W",
    "RV_M_dec": "RV_M",
    "CC_dec": "CC",
    "JC_dec": "JC",
    "PV_dec": "PV",
    "NV_dec": "NV",
    "SJ_dec": "SJ",
    "CQ": "CQ",
    "IVvol_daily_D_1_ATM": "IV_D1",
    "IVvol_daily_W_1_ATM": "IV_W1",
    "IVvol_daily_M_1_ATM": "IV_M1",
}
combo1 = combo1.rename(columns=rename_map)
 
keep_cols = [
    "date", "close", "log_return", "log_return_unshifted", "return", "return_unshifted",
    "RV", "RV_W", "RV_M",
    "CC", "JC", "PV", "NV", "SJ",
    "CQ",
    "IV_D1", "IV_W1", "IV_M1",
]
missing = [c for c in keep_cols if c not in combo1.columns]
if missing:
    raise ValueError(f"Missing columns in joint for combo1: {missing}")
 
combo1 = combo1[keep_cols].copy()
 
out_fp = DATA_DIR / "RV_IV_combo_1.csv"
combo1.to_csv(out_fp, index=False)
print("Saved:", out_fp)
print("Shape:", combo1.shape)
print("Columns:", combo1.columns.tolist())

Saved: C:\Users\livsem\master\tio4900_master_thesis\data\RV_IV_combo_1.csv
Shape: (3713, 18)
Columns: ['date', 'close', 'log_return', 'log_return_unshifted', 'return', 'return_unshifted', 'RV', 'RV_W', 'RV_M', 'CC', 'JC', 'PV', 'NV', 'SJ', 'CQ', 'IV_D1', 'IV_W1', 'IV_M1']
